# <font color=green> Global Warming Science</font>
#### https://courses.seas.harvard.edu/climate/eli/Courses/EPS101/

# Sea Level

### <font color=red> Please use the template below to answer the workshop questions. "XX" indicates places where you need to complete/write code or add a discussion.</font>

### your name:

### If not using FAS on-demand Jupyter server, required installations first:

from a terminal try: <br>
 conda install cartopy <br>
 pip install gsw <br>

Another option for cartopy: in Anaconda Navigator, click Environments, then change in the pull-down menu "Installed" to "All"; find cartopy, mark it for installation and click Apply.  

If both of these options do not work, try "pip install cartopy" instead of "conda install cartopy" for cartopy. 

In [ ]:
# import needed libraries and load data:
import numpy as np
import numpy.matlib as matlib
import pickle
import matplotlib.pyplot as plt
import scipy.integrate as integrate
import gsw
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import NaturalEarthFeature
from cartopy import config
from matplotlib.colors import BoundaryNorm
from matplotlib import ticker

# Read 2-d sea surface data for historical and rcp85 experiment
with open('./sea_level_variables.pickle', 'rb') as file:
    d = pickle.load(file)
    # print information about each extracted variable:
    for key in list(d.keys()):
        print(f"extracting pickled variable: name={key}; size={d[key].shape}")
        #print("type=",type(d[key]))
    globals().update(d)

### Explanation of input variables: 

**Sea level as a function of latitude and longitude for the first and last years of the historical and RCP8.5 scenarios:**

sealevel_historical, sealevel_historical_years

sealevel_rcp85, sealevel_rcp85_years

the corresponding lon/lat axes:

sealevel_lon, sealevel_lat

**Ocean temperatures as a function of depth, latitude, and longitude:**

Temperature_ocean_1850,
Temperature_ocean_2100,

and the corresponding three axes:

Temperature_ocean_lon,
Temperature_ocean_lat,
Temperature_ocean_lev,

Temperature_ocean_dZ: the thickness of temperature layers

**Global mean sea level time series and corresponding time axes:**

GMSL_thermosteric_historical,
GMSL_thermosteric_historical_years,

GMSL_thermosteric_rcp85,
GMSL_thermosteric_rcp85_years,

GMSL_since_1700_years,
GMSL_since_1700,


**Comments:**

1) sealevel_historical and sealevel_rcp8.5 both have zero spatial mean, and therefore represent only the deviation from this mean. The GMSL itself (contribution due to ocean warming) is given by GMSL_thermosteric_historical and GMSL_thermosteric_rcp85.

2) GMSL_since_1700 contains the Jevrejeva et al (2008) data

## 1) Characterizing sea level rise:

#### 1a) GMSL: Plot the globally-averaged sea level anomaly from 1850 to 2100 for historical and then RCP8.5 scenarios, and a quadratic polynomial to it. Discuss how and why the rate of the sea level rise in the two scenarios changes over this period.

In [ ]:
# combine historical and rcp85 records into one:
years=np.hstack([GMSL_thermosteric_historical_years,GMSL_thermosteric_rcp85_years]); 
SSH=np.hstack([GMSL_thermosteric_historical,GMSL_thermosteric_rcp85])

## least square fit of the time series to a parabola
# x=np.polyfit(XX,XX,2);
SSH_fit=x[0]*years**2+x[1]*years+x[2];

print(
    f"SSH_fit=x[0]*years**2+x[1]*years+x[2]; where "
    f"x={x[0]:g}, {x[1]:g}, {x[2]:g}"
)

# plot GMSL and fit:
plt.figure(dpi=200)
plt.plot(XX,XX,"b",lw=1,label="thermosteric historical")
plt.plot(XX,XX,"r",lw=1,label="thermosteric rcp8.5")
plt.plot(XX,XX,'g--',label="quadratic fit")
plt.xlabel('Year')
plt.ylabel('Sea level (m)');
plt.legend();
plt.title('GMSL');

### discussion:

XX

#### 1b) Spatial structure of sea level rise: Contour the spatial structure of sea level rise across the historical period and across the RCP8.5 period, describe and explain your results, in particular in the Southern Ocean. Read carefully the information about the workshop variables.

In [ ]:
# contour sea level changes across historical and rcp85 periods:
# preliminaries, defining configuration of subplots:
projection=ccrs.PlateCarree()
fig,axes=plt.subplots(1,2,figsize=(12,6)\
                      ,subplot_kw={'projection': projection},dpi=200)
cmap = plt.get_cmap('jet')

# plot historical GMSL:
axes[0].set_extent([0, 359.999, -90, 90], crs=ccrs.PlateCarree())
axes[0].coastlines(resolution='110m')
axes[0].gridlines()
c=axes[0].pcolormesh(sealevel_lon,sealevel_lat\
               ,(sealevel_historical[-1,:,:]-sealevel_historical[0,:,:])\
                +(GMSL_thermosteric_historical[-1]-GMSL_thermosteric_historical[0]) \
               ,vmin=-0.4,vmax=0.4\
                     ,cmap="bwr"\
              ,transform=ccrs.PlateCarree())
clb=plt.colorbar(c, shrink=0.45, pad=0.02,ax=axes[0],label="meter")
year_start = round(sealevel_historical_years[0])
year_end = round(sealevel_historical_years[-1])
axes[0].set_title("sealevel rise over years "+repr(year_end)
                                      +"-"+repr(year_start))

# plot rcp8.5 GMSL:
axes[1].set_extent([0, 359.999, -90, 90], crs=ccrs.PlateCarree())
XX
axes[1].set_title(f"Sealevel rise over years {year_end}-{year_start}")

plt.subplots_adjust(top=0.92, bottom=0.08, left=0.01, right=0.95 \
                    ,hspace=0.15,wspace=0.1)
plt.show();

## 2) Temperature, density and sea level rise.

#### 2a) Plot ocean water density as function of temperature for $T=-2,\ldots,30$ °C. Plot the expansion coefficient $\alpha$ ($^\circ$C$^{-1}$) for the same temperature range. Discuss the implications on sea level rise of the dependence of the expansion coefficient on temperature.

In [ ]:
T=np.arange(-2,30,0.1)
S=T*0+35
P=T*0
rho=gsw.rho(S,T,P)
# calculate the thermal expansion coefficient at the specified salinity, 
# temperature and pressure:
alpha=gsw.alpha(S,T,P)

print(
    f"alpha(2,10)={XX:g} {XX:g} "
    f"rho(3-2,11-10)={XX-XX:g} "
    f"{XX-XX:g}"
)
plt.figure(figsize=(8,4),dpi=150)
plt.subplot(1,2,1)
plt.plot(XX,XX)
plt.xlabel("Temperature (C)")
plt.ylabel("density (kg/m$^3$)")

plt.subplot(1,2,2)
plt.plot(XX,XX)
plt.xlabel("Temperature (C)")
plt.ylabel("alpha (C$^{-1}$)")

plt.tight_layout()
plt.show();

<b>Discussion:</b> XX.

#### 2b) Assuming warming is exponential in depth, $\delta T(z)=4\exp(z/500)$, and extending to the ocean bottom at 4 km. Calculate the expected sea level rise. Assume the equation of state is $\rho=\rho_0\left[1-\alpha\left(T-T_0\right)\right]$, in which $\rho_0=1025$ kg/m$^3$, $\alpha=1.668\times 10^{-4}$ °C$^{-1}$ and $T_0=10$ °C. What is the expected sea level rise if the exponential decay scale is 1000 m instead of 500 m? Explain.

In [ ]:
# calculate the expected GMSL rise:
h=4000.0; # average depth of the ocean
alpha=XX; # thermal expansion coefficient at T=10C, [alpha is approximated to be a constant here]
decay_scale=500
N=101;
z=np.arange(-h,0,h/N)
Delta_T=XX

# solve using two different approaches:

# (1) numerical integration using a library function:
Delta_GMSL=integrate.quad(XX)
print(
    f"The warming will lead to {Delta_GMSL[0]:5.3g} meters of sea level "
    f"rise (library integration)."
)

# (2) analytic integration: Delta_GMSL=\integral_{-h}^0 alpha*Delta_T(z)dz
Delta_GMSL1=XX
print(
    f"The warming will lead to {Delta_GMSL1:5.3g} meters of sea level "
    f"rise (analytical)."
)

# plot the warming temperature profile:
plt.figure(dpi=200)
plt.plot(XX,XX)
plt.xlabel("$\\Delta T(z)$, warming ($^\\circ$C)")
plt.ylabel("Depth (m)")
plt.show();

In [ ]:
# calculate the expected GMSL rise again, with a decay_scale of 1000 m:
XX

# plot the warming temperature profile:
XX

#### 2c) Sea ice and sea level rise: 
Consider a 1 cm$^3$ ice cube floating in a cup of seawater whose radius is 5 cm and where the water height is 10 cm. Calculate the change in water level in the cup when the ice cube melts, assuming first that the seawater is of salinity of 35 ppt, and then when the cup is filled with fresh water. Assume that the melt water mixes with the water in the cup and that the density is linearly related to salinity as $\rho(T_0,S)=\rho_0(1+\alpha_S(S-S_0))$ with $S_0=35$ ppt, $\rho_0=1$ gr/cm$^3$, and $\alpha_S=0.8$ ppt$^{-1}$.

The following website is helpful conceptually:
https://sealevel.nasa.gov/news/261/melting-ocean-ice-affects-sea-level-unlike-ice-cubes-in-a-glass/

In [ ]:
# Given
S0 = 35           # ppt
rho0 = 1025       # kg m^-3
rho_ice = XX      # kg m^-3
alpha_S = 0.8     # ppt^-1
radius = 0.05     # m
height = 0.10     # m
cube_size=XX      # m
d0 = height       # m

# calculate the mass of ice present in the ice cube
mass_melt = XX

# Calculate the initial mass occupied by the water in the cup: 
# (calculate it from the water volume, but take into account the 
# water displaced by the ice)
mass_water = XX

# calculate the new salinity by taking a mass weighted average of the water in the cup:
S_new = XX

# Calculate the new density of the water due to this change in salinity
rho_new = XX

# use the equation for change in depth with change in density
deltad = XX

print(
    "If you neglect the water displaced originally by the ice cube, "
    f"\nthe height change is {deltad:g} cm"
)

# Now perform the same calculation assuming the cup is filled with freshwater
XX

#### (2d) Optional extra credit: Given a certain amount of heat (in J) added to the climate system, would it cause more sea level rise if it melts land ice or leads to expansion of seawater? Your answer should take into account the initial temperatures of the ice and of the ocean. Contour the ratio of the two possible contributions to sea level rise as a function of these two temperatures.

In [ ]:
# available heat: J/m^2
Heat=1000

# initial temperatures:
T_i_ocean=0
T_i_ice=-5

# latent heat of freezing and specific heat capacities:
L=334e3 # J/kg 
c_p_ice=XX # J/(kg C)
c_p_water=XX # J/(kg C)

# densities:
rho_ice=900
rho_water=1024
# thermal expansion coefficient at the freezing temperature:
alpha=gsw.alpha(35,0,0)

# sea level rise due to ocean warming:
Delta_SSH_ocean_warming=XX

# sea level rise due to ice melting:
heat_ice_to_melt_temperature=XX
heat_for_melting=Heat-heat_ice_to_melt_temperature
Delta_SSH_ice_melting=XX

print(
    f" SSH due to ocean warming={Delta_SSH_ocean_warming:g}"
    f"\n SSH due to ice melting={Delta_SSH_ice_melting:g}"
)


## 3) Given the 3d temperature at 1850 and the CMIP5 RCP8.5 estimate at 2100, calculate and plot the local contribution to sea level rise as function of longitude and latitude, and calculate the expected GMSL rise. Use appropriate latitude-dependent weighting for the averaging. Assume that salinity is a constant at 35 ppt, considering only the effects of warming.

In [ ]:
# calculate ocean densities before and after the warming:
Salinity=np.zeros(Temperature_ocean_1850.shape)+35
theta1=Temperature_ocean_1850
pressure=np.zeros(theta1.shape)
rho1=gsw.rho(Salinity,theta1,pressure)

theta2=Temperature_ocean_2100
rho2=gsw.rho(Salinity,theta2,pressure)

# calculate the sea level rise at each horizontal location:
density_ratio=XX
Nz,Ny,Nx=density_ratio.shape

# Integrate the appropriate density ratio in the vertical to find dSSH locally:
dSSH=np.zeros((Ny,Nx))
for i in range(Nx):
    for j in range(Ny):
        for k in range(Nz):
            # integrate over non-NaN values only!
            if ~np.isnan(density_ratio[k,j,i]):
                dSSH[j,i]=dSSH[j,i]+XX

# calculate global averaged GMSL rise using weighting by the variable arealcello:
R= XX # Earth radius
pi=XX
dlat=np.abs(Temperature_ocean_lat[1]-Temperature_ocean_lat[0])*pi/180
dlon=XX
weight=R**2*dlat*dlon*np.cos(Temperature_ocean_lat*pi/180)
nlat, nlon = dSSH.shape
weight2 = np.tile(weight[:, None], (1, nlon))   # shape (nlat, nlon)
dGMSL=XX # use np.nansum
print(f"GMSL rise due to warming is {dGMSL:5.3g} m")

## make a pcolormesh figure of the pattern of SSH anomaly
# contour sea level changes across historical and rcp85 periods:
# preliminaries, defining configuration of subplots:
projection=ccrs.PlateCarree()
fig,axes=plt.subplots(1,1,figsize=(6,6)\
                      ,subplot_kw={'projection': projection},dpi=200)
XX add plot commands here
axes.set_title("Local contributions to sea level rise due to warming");

## 4) Wind forcing and the Coriolis force:

### a) Consider a storm in the Adriatic Sea north of Monte Gargano, with wind blowing along the length of the sea toward Venice at 30 m/s. Calculate the sea level rise at the far north end of the sea.

In [ ]:
# define parameters and variables to be use
Rhoa=XX;  # kg/m^3, density of air
Cd=0.0013;  # bulk formula coefficient
U=XX;    # m/s, wind speed
U_km_per_hour=XX
rho0=XX;  # kg/m^3, density of sea water
g=XX;  # m/s^2, acceleration due to gracity
L=460000; # m, the length of the northern Adriatic Sea in meters
h=50; # the approx average depth of the Adriatic Sea north of Monte Gargano, in meters

# calculate wind stress using the bulk formula which says: wind stress=Cd*[atmospheric density]*[surface wind squared]
tau=XX
# calculate the sea level difference along the whole length of the Adriatic sea
dH=XX

# print results:
print(
    f"given a wind speed of U= {U:3.3g} m/s "
    f"({U_km_per_hour:3.3g} km/hour), the stress is tau= "
    f"{tau:3.2g} N/m^2,"
)
print(
    "and the sea level rise in the northern edge of the Adriatic Sea is expected"
    f" to be {dH:4.2g} meters."
);

### (b) Calculate the range of sea level changes expected in the center of a category 4 hurricane whose center pressure is 920 to 944 mb.

XX Perform the analytic calculation here

In [ ]:
print(f"delta h for 920 mb={XX:g} meters")
print(f"delta h for 944 mb={XX:g} meters")

### c) Calculate the change in sea level difference across the Gulf Stream at a latitude 30N in response to a weakening of the current by 10%. Assume the original flow was 1 m/s, and that the width of the Gulf Stream there is 50 km.

XX Perform the analytic calculation here

In [ ]:
# and print the results here:
print("delta h=",XX,"meters")

## 5) Sea-level gravitational fingerprint: 

Consider the sea level fingerprints in figure 4.7a,b. Where is melting happening? Explain these sea level change patterns.

XX

## 6) Optional extra credit: Decadal variability vs climate change. 

Reproduce Figure 2 of Jevrejeva et al. (2008) using their data that is available here as the GMSL since 1700. Discuss: what are the implications of your analysis on the interpretation of the sea level acceleration during the most recent 20 years presented in Figure 1 of Nerem et al. (2018)? (see bibliography section of course notes) 

In [ ]:
XX